# 03 · FlashSAC 심화: bootstrap critic norm 안정화

목표: 자기 예측을 target에 사용하는 toy linear critic을 만들고, 큰 learning rate에서 weight norm이 커지는 현상과 unit-norm projection의 효과를 비교합니다.

주의: 이 실험은 FlashSAC architecture와 성능을 재현하지 않습니다. 논문의 안정화 직관 하나를 격리한 교육용 proxy입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(23)

## 1. 합성 transition

상태 feature $x$, 다음 상태 feature $x'$, reward $r$을 만듭니다. Critic은 $Q_w(x)=x^T w$이고 target은 $r+\gamma Q_w(x')$입니다. 실제 SAC와 달리 target network, double Q, entropy 항을 생략해 불안정성을 일부러 드러냅니다.

In [ ]:
samples, features = 4096, 32
x = rng.normal(size=(samples, features))
transition = np.eye(features) * 1.02 + rng.normal(0, 0.015, (features, features))
x_next = x @ transition + rng.normal(0, 0.1, (samples, features))
true_w = rng.normal(0, 0.2, size=features)
reward = x @ true_w + rng.normal(0, 0.1, size=samples)
gamma = 0.995
print(x.shape, x_next.shape, reward.shape)

## 2. Plain critic과 projected critic

두 critic은 같은 mini-batch를 봅니다. Projected 버전만 각 update 뒤 weight를 unit sphere에 투영합니다. 논문의 actor/critic 전체는 더 복잡하며 normalization parameter에는 $\sqrt d$ norm을 사용합니다.

In [ ]:
def train(project_weights, steps=500, batch_size=256, learning_rate=0.08):
    w = rng.normal(0, 0.05, size=features)
    norms, losses, grad_norms = [], [], []
    for _ in range(steps):
        ids = rng.integers(0, samples, size=batch_size)
        xb, xnb, rb = x[ids], x_next[ids], reward[ids]

        # 교육용으로 target을 detach한 것처럼 현재 step의 고정값으로 취급합니다.
        target = rb + gamma * (xnb @ w)
        error = xb @ w - target
        gradient = 2.0 * xb.T @ error / batch_size
        w = w - learning_rate * gradient

        if project_weights:
            w = w / max(np.linalg.norm(w), 1e-12)

        norms.append(np.linalg.norm(w))
        losses.append(np.mean(error**2))
        grad_norms.append(np.linalg.norm(gradient))
    return np.array(norms), np.array(losses), np.array(grad_norms)

plain = train(project_weights=False)
projected = train(project_weights=True)
print(f'plain final weight norm: {plain[0][-1]:.3f}')
print(f'projected final weight norm: {projected[0][-1]:.3f}')
assert np.allclose(projected[0], 1.0, atol=1e-8)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
labels = ('weight norm', 'Bellman loss', 'gradient norm')
for ax, index, label in zip(axes, range(3), labels):
    ax.plot(plain[index], label='plain')
    ax.plot(projected[index], label='unit-norm projection')
    ax.set_title(label)
    ax.set_xlabel('gradient step')
    ax.set_yscale('log' if index > 0 else 'linear')
    ax.legend()
plt.tight_layout();

## 3. 결과 해석

- Projection은 weight norm을 정확히 제한하지만 loss를 자동으로 최소화하지는 않습니다.
- 실제 FlashSAC은 residual block, BatchNorm, RMSNorm, cross-batch prediction, distributional critic과 reward scaling을 함께 사용합니다.
- 따라서 이 실험에서 projection이 좋아 보인다고 해서 논문 구성 요소의 개별 인과효과가 증명되는 것은 아닙니다.
- Target network와 clipped double Q를 추가하면 target이 느리게 움직이고 낙관 오차가 줄어듭니다.

## 심화 과제

1. 별도의 target weight를 만들고 $\bar w\leftarrow\tau w+(1-\tau)\bar w$를 구현합니다.
2. 두 critic 중 작은 값을 쓰는 clipped double-Q target을 추가합니다.
3. batch size와 update 횟수의 곱을 고정하고 `큰 batch + 적은 update`와 `작은 batch + 많은 update`를 비교합니다.
4. weight norm뿐 아니라 feature norm과 gradient norm의 percentile을 기록합니다.
5. 공식 FlashSAC 코드의 CPU simulator 설정으로 옮길 때 commit, seed, raw return, wall-clock 측정 범위를 기록합니다.